# 🎮 Pipeline Tự Động Hóa Tạo Hình Nhân Vật Game 2D $\rightarrow$ 3D Chuẩn UV Texture 2048

**Nghiên cứu ứng dụng:** Kết hợp mô hình khuếch tán hình ảnh (*Stable Diffusion*) và mô hình tái tạo không gian có nướng vân bề mặt UV (*TripoSR Texture Baking 2K + Taubin Mesh Smoothing*) phục vụ tự động hóa khâu Concept-to-Asset trong sản xuất Game.

> 💡 **Đặc điểm nổi bật của phiên bản này:**
> 1. **Nướng Texture UV Atlas 2048x2048:** Phủ kín màu 360 độ toàn bộ cơ thể (cả mặt trước và mặt sau), không còn bị loang lổ hay mất màu.
> 2. **Bộ lọc Taubin Filter:** Làm mượt bề mặt đa giác mịn màng, xóa bỏ hiện tượng răng cưa khối.
> 3. **Lọc khối độc lập (Mesh Cleaning):** Tự động loại bỏ hoàn toàn các mảnh vụn, đốm trắng hoặc khối u thừa ở vai và gáy.
> 4. **Bảng số liệu khoa học:** Tự động đo đạc Latency, Vertices, Faces, VRAM và dung lượng GLB cho bài báo nghiên cứu.

In [ ]:
# =============================================================================
# BƯỚC 1: Cài đặt Môi trường & Thư viện Tối ưu (Chuẩn hóa từ test_paper.ipynb)
# =============================================================================
import subprocess, sys, os

print("⚡ [1/4] Kiểm tra phần cứng GPU...")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

print("\n⚡ [2/4] Cài đặt công cụ biên dịch scikit-build-core & pybind11...")
!pip install -q scikit-build-core pybind11

print("\n⚡ [3/4] Tải mã nguồn TripoSR & cài đặt thư viện phụ thuộc nới lỏng...")
if not os.path.exists("/content/TripoSR"):
    !git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR

# Nới lỏng phiên bản requirements để tránh xung đột
!sed -E 's/[=<>!~].*//' /content/TripoSR/requirements.txt | grep -v -i -E 'torch|torchmcubes' > /content/req_loose.txt
!pip install -q -r /content/req_loose.txt
!pip install -q --no-build-isolation git+https://github.com/tatsy/torchmcubes.git

print("\n⚡ [4/4] Cài đặt thư viện AI, xử lý ảnh & giao diện trực quan...")
!pip uninstall -y -q onnxruntime-gpu onnxruntime
!pip install -q onnxruntime rembg trimesh gradio diffusers transformers accelerate

print("\n✅ CÀI ĐẶT MÔI TRƯỜNG THÀNH CÔNG! SẴN SÀNG CHẠY THỰC NGHIỆM.")


In [ ]:
# =============================================================================
# BƯỚC 2: Khởi tạo Mô hình Tạo Ảnh Concept 2D (Stable Diffusion)
# =============================================================================
import torch, time
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Đang nạp mô hình Stable Diffusion trên {device}...")

model_id = "runwayml/stable-diffusion-v1-5"
pipe_2d = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None
)
pipe_2d.scheduler = DPMSolverMultistepScheduler.from_config(pipe_2d.scheduler.config)
pipe_2d.to(device)
print("✅ Mô hình 2D Concept Art đã sẵn sàng!")

def generate_2d_character(prompt_desc, negative_prompt="", steps=25, guidance=7.5):
    """
    Sinh ảnh concept nhân vật Game tối ưu cho dựng 3D (A-pose, nền trắng, toàn thân)
    """
    words = prompt_desc.strip().split()
    clean_desc = " ".join(words[:40]) if len(words) > 40 else prompt_desc.strip()
    
    full_prompt = (
        f"full body concept art of {clean_desc}, "
        "front view, centered, standing A-pose, neutral pose, white solid background, "
        "game character design, clean edges, flat lighting, high quality, 8k resolution"
    )
    neg_prompt = (
        "cropped, cut off, bad anatomy, deformed limbs, blurry, noise, "
        "complex background, shadows on background, dark background, side view, back view, text, watermark "
        + negative_prompt
    )
    
    image = pipe_2d(
        prompt=full_prompt,
        negative_prompt=neg_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        height=512,
        width=512
    ).images[0]
    return image


In [ ]:
# =============================================================================
# BƯỚC 3: Pipeline Tái Tạo 3D: Bake Texture 2048 + Taubin Filter + Mesh Clean
# =============================================================================
import os, glob, time, subprocess, shutil
import trimesh
from trimesh.smoothing import filter_taubin
from rembg import remove
from PIL import Image

OUT_DIR = "/content/outputs_3d"
os.makedirs(OUT_DIR, exist_ok=True)

def convert_2d_to_3d_pipeline(
    pil_image_or_path,
    output_dir=OUT_DIR,
    mc_resolution=384,
    texture_resolution=2048,
    smooth_iterations=10
):
    """
    Quy trình chuẩn kết hợp từ test_paper.ipynb:
    1. Tách nền, cắt sát viền vật thể và đệm vuông cân đối (1.2x).
    2. Chạy TripoSR với Marching Cubes 384 và nướng Texture UV Atlas 2048x2048 phủ kín 360 độ.
    3. Hậu kỳ làm sạch lưới: giữ khối lớn nhất (loại bỏ đốm rác), làm mượt Taubin Filter và xuất file .GLB sắc nét.
    """
    step_start = time.time()
    task_id = f"char_{int(time.time()*1000)}"
    task_dir = os.path.join(output_dir, task_id)
    os.makedirs(task_dir, exist_ok=True)
    
    # 1. Nạp và xử lý tiền kỳ ảnh 2D
    if isinstance(pil_image_or_path, str):
        img_raw = Image.open(pil_image_or_path).convert("RGBA")
    else:
        img_raw = pil_image_or_path.convert("RGBA")
        
    print("✂️ [1/3] Đang tách nền và căn chỉnh khung hình chuẩn...")
    img_nobg = remove(img_raw)
    bbox = img_nobg.getbbox()
    if bbox:
        img_nobg = img_nobg.crop(bbox)
        
    # Đệm thành hình vuông cân đối 1.2 lần (tương đương foreground ratio tối ưu ~0.83)
    s = int(max(img_nobg.size) * 1.2)
    canvas = Image.new("RGBA", (s, s), (255, 255, 255, 0))
    canvas.paste(img_nobg, ((s - img_nobg.width) // 2, (s - img_nobg.height) // 2), img_nobg)
    
    clean_img_path = os.path.join(task_dir, "clean_input.png")
    canvas.save(clean_img_path)
    
    # 2. Chạy TripoSR với cơ chế Texture Baking 2048
    print(f"⚡ [2/3] Đang tái tạo khối 3D (MC={mc_resolution}) & Nướng Texture UV 2K ({texture_resolution}x{texture_resolution})...")
    t_recon_start = time.time()
    cmd = [
        "python", "run.py", clean_img_path,
        "--output-dir", task_dir,
        "--no-remove-bg",
        "--mc-resolution", str(mc_resolution),
        "--bake-texture",
        "--texture-resolution", str(texture_resolution)
    ]
    res = subprocess.run(cmd, cwd="/content/TripoSR", capture_output=True, text=True)
    if res.returncode != 0:
        print("❌ Lỗi khi chạy TripoSR:\n", res.stderr[-2000:])
        raise RuntimeError("TripoSR gặp lỗi trong quá trình nướng texture.")
        
    time_recon = round(time.time() - t_recon_start, 2)
    print(f"✅ Tái tạo 3D & nướng texture xong trong {time_recon} giây!")
    
    # 3. Hậu kỳ dọn dẹp lưới & làm mượt bề mặt
    print("🧹 [3/3] Đang dọn dẹp khối đa giác & làm mượt Taubin Filter...")
    obj_files = glob.glob(f"{task_dir}/**/mesh.obj", recursive=True)
    if not obj_files:
        raise FileNotFoundError("Không tìm thấy tệp mesh.obj trong thư mục đầu ra.")
        
    mesh_path = obj_files[0]
    m = trimesh.load(mesh_path, force="mesh")
    
    # Giữ lại khối đa giác lớn nhất (loại bỏ hoàn toàn mảnh vụn và khối u thừa ở vai/gáy)
    parts = m.split(only_watertight=False)
    if len(parts) > 0:
        m = max(parts, key=lambda p: p.area)
        
    # Áp dụng bộ lọc Taubin để làm mượt đa giác mà không làm teo hình dáng
    if smooth_iterations > 0:
        try:
            filter_taubin(m, iterations=smooth_iterations)
        except Exception as e:
            print(f"⚠️ Bỏ qua filter_taubin: {e}")
            
    final_glb = os.path.join(task_dir, "character_clean.glb")
    m.export(final_glb)
    
    total_time = round(time.time() - step_start, 2)
    num_vertices = len(m.vertices)
    num_faces = len(m.faces)
    file_size_kb = round(os.path.getsize(final_glb) / 1024, 2)
    
    stats = {
        "vertices": num_vertices,
        "faces": num_faces,
        "file_size_kb": file_size_kb,
        "time_total": total_time,
        "time_recon": time_recon,
        "glb_path": final_glb,
        "clean_img_path": clean_img_path
    }
    return final_glb, stats

print("✅ Pipeline TripoSR Bake Texture + Taubin Filter đã sẵn sàng!")


In [ ]:
# =============================================================================
# BƯỚC 4: Chạy Thực Nghiệm Mẫu & Đo Đạc Chỉ Số Khoa Học (Cho Bài Báo)
# =============================================================================
sample_prompt = "Vietnamese soldier in 1972 uniform, heroic standing pose, game character"

print(f"🚀 BẮT ĐẦU CHẠY THỰC NGHIỆM TẠO NHÂN VẬT: '{sample_prompt}'\n")
pipeline_start = time.time()

# 1. Sinh Concept 2D
t0 = time.time()
concept_2d = generate_2d_character(sample_prompt, steps=25)
time_2d = round(time.time() - t0, 2)
concept_2d.save("/content/concept_sample.png")
print(f"✅ [Giai đoạn 1] Đã tạo xong Concept 2D: {time_2d} giây")

# 2. Tái tạo 3D & Nướng Texture 2K
glb_file, metrics = convert_2d_to_3d_pipeline(
    concept_2d,
    mc_resolution=384,
    texture_resolution=2048,
    smooth_iterations=10
)

# 3. Đo VRAM sử dụng
vram_peak = round(torch.cuda.max_memory_allocated() / (1024**3), 2) if torch.cuda.is_available() else 0

print("\n" + "="*65)
print("📊 BẢNG SỐ LIỆU ĐO ĐẠC KHOA HỌC (SCIENTIFIC METRICS CHO LUẬN VĂN/BÀI BÁO):")
print("="*65)
print(f"• Thời gian tạo Concept Art 2D (Stable Diffusion) : {time_2d} s")
print(f"• Thời gian Tái tạo 3D & Nướng Texture 2K         : {metrics['time_recon']} s")
print(f"• TỔNG THỜI GIAN TOÀN BỘ PIPELINE (End-to-End)   : {round(time.time() - pipeline_start, 2)} s")
print(f"• Độ phân giải Marching Cubes                     : 384")
print(f"• Độ phân giải Texture UV Map                     : 2048 x 2048")
print(f"• Số lượng đỉnh (Vertices Count)                 : {metrics['vertices']:,}")
print(f"• Số lượng mặt đa giác (Faces Count)             : {metrics['faces']:,}")
print(f"• Dung lượng tệp xuất xưởng (.GLB)               : {metrics['file_size_kb']} KB")
print(f"• Bộ nhớ GPU tiêu thụ (Peak VRAM Usage)          : {vram_peak} GB / 16.0 GB")
print(f"• Tệp kết quả GLB đã làm sạch                    : {glb_file}")
print("="*65)

display(concept_2d)


In [ ]:
# =============================================================================
# BƯỚC 5: Giao Diện Trực Quan (Gradio Web UI) - Trình Diễn & Xuất Asset 3D
# =============================================================================
import gradio as gr

def full_pipeline_gradio(prompt_text, custom_image, mc_res, tex_res, smooth_iter):
    start = time.time()
    if custom_image is not None:
        img_input = custom_image
        proc_info = "Sử dụng ảnh do người dùng tải lên trực tiếp."
    else:
        if not prompt_text or not prompt_text.strip():
            prompt_text = "Vietnamese soldier in 1972 uniform, heroic standing pose"
        img_input = generate_2d_character(prompt_text)
        proc_info = f"Sinh ảnh 2D từ Prompt: '{prompt_text}'"
        
    glb_out, stats = convert_2d_to_3d_pipeline(
        img_input,
        mc_resolution=int(mc_res),
        texture_resolution=int(tex_res),
        smooth_iterations=int(smooth_iter)
    )
    total_time = round(time.time() - start, 2)
    
    metrics_md = f"""
### 📈 Số Liệu Thực Nghiệm (Scientific Metrics):
- **Thời gian xử lý tổng cộng:** `{total_time} giây` (2D: `{round(total_time - stats['time_recon'], 2)}s` | 3D + Texture: `{stats['time_recon']}s`)
- **Độ phân giải Marching Cubes:** `{int(mc_res)}`
- **Độ phân giải Texture Atlas:** `{int(tex_res)} x {int(tex_res)} (Bake UV)`
- **Số mặt đa giác (Faces):** `{stats['faces']:,}`
- **Số đỉnh (Vertices):** `{stats['vertices']:,}`
- **Kích thước file GLB:** `{stats['file_size_kb']} KB`
- **Làm mượt đa giác:** `Bộ lọc Taubin Filter ({int(smooth_iter)} vòng)`
- **Ghi chú quy trình:** {proc_info}
    """
    return img_input, glb_out, metrics_md

with gr.Blocks(title="AI Game Character Generator Pro") as demo:
    gr.Markdown("""
    # 🎮 Pipeline AI Tạo Hình Nhân Vật Game: 2D $\\rightarrow$ 3D Chuẩn UV Texture 2048
    **Nghiên cứu ứng dụng:** Kết hợp mô hình khuếch tán (*Stable Diffusion*) và mô hình tái tạo không gian có nướng vân bề mặt UV (*TripoSR Texture Baking 2K + Taubin Smoothing*) để tự động hóa khâu Concept-to-Asset trong sản xuất Game.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_box = gr.Textbox(
                label="1. Nhập mô tả nhân vật (Text Prompt)",
                placeholder="Ví dụ: Vietnamese soldier in 1972 uniform, cyberpunk ninja, medieval knight...",
                lines=3
            )
            image_box = gr.Image(
                label="Hoặc tải lên ảnh 2D có sẵn (Tùy chọn)",
                type="pil"
            )
            with gr.Accordion("⚙️ Cấu hình chất lượng mô hình (Texture & Geometry)", open=True):
                mc_slider = gr.Slider(minimum=256, maximum=384, value=384, step=32, label="Độ phân giải Marching Cubes", info="384 cho lưới chi tiết và hình khối rõ nét nhất")
                tex_slider = gr.Radio(choices=[1024, 2048], value=2048, label="Độ phân giải Texture UV Atlas", info="2048x2048 phủ kín màu 360 độ mặt trước và mặt sau")
                smooth_slider = gr.Slider(minimum=0, maximum=20, value=10, step=2, label="Độ mượt Taubin Filter", info="Khử răng cưa và khối u thừa mà không làm teo nhân vật")
                
            btn_generate = gr.Button("🚀 Chạy Toàn Bộ Pipeline (2D + 3D)", variant="primary")
            
        with gr.Column(scale=1):
            preview_2d = gr.Image(label="Ảnh Concept 2D (Giai đoạn 1)")
            preview_3d = gr.Model3D(label="Mô Hình 3D UV Texture 360° (Giai đoạn 2)")
            display_metrics = gr.Markdown()
            
    btn_generate.click(
        fn=full_pipeline_gradio,
        inputs=[prompt_box, image_box, mc_slider, tex_slider, smooth_slider],
        outputs=[preview_2d, preview_3d, display_metrics]
    )

demo.launch(theme=gr.themes.Soft(), share=True, debug=True)
